# 07-units-T — Dimensionally-constrained ozone search WITH temperature in the pool (Bogotá)

Follow-up to `07_o3_units_equation_discovery.ipynb`. That run used the original no-lag selection and
**never offered `temp_2m`**, so it could not say anything about temperature: the earlier wording "the constrained
equation keeps PM2.5 but drops T" was wrong. What stops the free-form lead equation (Eq. 2,
`T/(NO/PM2.5 + 0.634 + w700) + sqrt(NO2*SW/(0.483*RH))`) from passing the unit check is `w700` (Pa/s) being added to a
ratio and a constant with no scale constant of its own.

**One change only** relative to the original constrained run: `temp_2m` (unit `K`, a temperature *difference*, as in the
units audit, notebook A4) is added to the candidate pool. Everything else is identical: the same pool
(`photochemical_potential` decomposed into `no2` and `radiation_mean`), the same units, the same operators, `niterations=200`,
`populations=24`, `maxsize=25`. Symbolic search is stochastic, so it is run for several seeds.

Scoring (same protocol as the other lead equations, notebook A2/A4): leave-one-year-out, constants refitted by non-linear least
squares on the training years, **both** the pooled R2 and the mean of per-year R2, on the same 2919 days as Eq. 2.

In [1]:
# %% Settings (edit here, then Run All)
SEEDS = [0, 1, 2]                      # symbolic search is stochastic: report all of them
PYSR_KW = dict(niterations=200, binary_operators=["+", "-", "*", "/"], unary_operators=["log", "sqrt", "exp"],
               model_selection="best", maxsize=25, populations=24)     # identical to 07_o3_units_equation_discovery
ADD_TEMPERATURE = True                 # temperature offered, as in the previous constrained run
OFFER_PM25 = False                  # True: PM2.5 offered as well (tests whether the search keeps it now that the selection no longer does)
# 2026-10-20 (corrected fire file): the selection is photochemical_potential, fire E lag 2, fog_stagnation, wind speed, no_ratio, w700, radiation.
# Compound variables are split into raw factors (as before): photochemical_potential = NO2 x SW -> no2 + radiation_mean;
# no_ratio = NO2/NO -> no2 + no; fog_stagnation = RH/(1+u) -> rh_2m (+ wind speed, already selected).
UNIT_MAP = {'no': 'kg/m^3', 'pm25': 'kg/m^3', 'no2': 'kg/m^3', 'rh_2m': '', 'w700': 'Pa/s', 'wind_speed_era5': 'm/s', 'hotspots_sector_E_lag2': '',
            'radiation_mean': 'W/m^2', 'temp_2m': 'K'}            # degC treated as a temperature difference (dimension Theta), as in notebook A4
Y_UNITS = 'kg/m^3'                     # same documented mass-concentration assumption as the PM2.5 notebook
MAX_COMPLEXITY_REPORTED = 25
TAG = "units_fire2"                     # output file stem


In [2]:
import sys, json, re, warnings
from pathlib import Path
PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
import numpy as np, pandas as pd, sympy as sp
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / "bogota"
df = pd.read_parquet(DATA_DIR / "bogota_o3_master_dataset.parquet")
df["date"] = pd.to_datetime(df["date"]); df["year"] = df["date"].dt.year
df = df.dropna(subset=['o3']).reset_index(drop=True)
selection = json.load(open(DATA_DIR / "bogota_o3_feature_selection.json"))
features = ['no2', 'no', 'radiation_mean', 'hotspots_sector_E_lag2', 'rh_2m', 'wind_speed_era5', 'w700']   # the selected variables, compound ones split into raw factors
if ADD_TEMPERATURE: features.append('temp_2m')
if OFFER_PM25: features.append('pm25')
print("pool:", features)
data = df.dropna(subset=features + ['o3']).reset_index(drop=True)
X = data[features].to_numpy(); y = data['o3'].to_numpy()
X_units = [UNIT_MAP[f] for f in features]
print(data.shape, dict(zip(features, X_units)))

pool: ['no2', 'no', 'radiation_mean', 'hotspots_sector_E_lag2', 'rh_2m', 'wind_speed_era5', 'w700', 'temp_2m']
(2917, 220) {'no2': 'kg/m^3', 'no': 'kg/m^3', 'radiation_mean': 'W/m^2', 'hotspots_sector_E_lag2': '', 'rh_2m': '', 'wind_speed_era5': 'm/s', 'w700': 'Pa/s', 'temp_2m': 'K'}


In [3]:
from pysr import PySRRegressor
fronts = {}
for seed in SEEDS:
    sr = PySRRegressor(**PYSR_KW, random_state=seed)
    sr.fit(X, y, variable_names=features, X_units=X_units, y_units=Y_UNITS)
    f = sr.equations_[['complexity', 'loss', 'equation']].copy(); f['seed'] = seed
    f.to_csv(DATA_DIR / f"bogota_o3_pysr_{TAG}_pareto_front_seed{seed}.csv", index=False)
    fronts[seed] = f
    print("seed", seed, "done; best:", sr.get_best()["equation"])

Compiling Julia backend...


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(



Expressions evaluated per second: 9.800e+03
Progress: 22 / 200 total iterations (11.271%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.894e+01  0.000e+00  y[m⁻³ kg] = 20.146[?]
3           4.142e+01  8.349e-02  y[m⁻³ kg] = radiation_mean[m⁻² W] * 0.10531[?]
4           4.017e+01  3.046e-02  y[m⁻³ kg] = sqrt(radiation_mean[m⁻² W]) * 1.4799[?...
                                              ]
5           3.603e+01  1.089e-01  y[m⁻³ kg] = (hotspots_sector_E_lag2 * 0.0071582[?]...
                                              ) + 18.078[?]
8           3.001e+01  6.091e-02  y[m⁻³ kg] = sqrt((radiation_mean[m⁻² W] * 1.6172[?...
                                              ]) - (hotspots_sector_E_lag2 * -0.34631[?]))
10          2.990e+01  1.868e-03  y[m⁻³ kg] = sqrt((radiation_mea

┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


seed 0 done; best: sqrt(sqrt((9.222794 - (sqrt(hotspots_sector_E_lag2) / (no * -0.0199888))) * (radiation_mean * no2)))


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:
/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(



Expressions evaluated per second: 1.090e+04
Progress: 25 / 200 total iterations (12.667%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.894e+01  0.000e+00  y[m⁻³ kg] = 20.151[?]
3           4.142e+01  8.346e-02  y[m⁻³ kg] = radiation_mean[m⁻² W] * 0.10549[?]
4           4.017e+01  3.056e-02  y[m⁻³ kg] = sqrt(radiation_mean[m⁻² W] * 2.1977[?]...
                                              )
5           3.602e+01  1.092e-01  y[m⁻³ kg] = (hotspots_sector_E_lag2 / 133.43[?]) +...
                                               18.042[?]
7           3.435e+01  2.360e-02  y[m⁻³ kg] = sqrt(sqrt(hotspots_sector_E_lag2 / 0.0...
                                              1746[?])) + 10.656[?]
8           3.085e+01  1.075e-01  y[m⁻³ kg] = sqrt((hotspots_sector_E_lag2 * 0.46253...
   


Expressions evaluated per second: 1.040e+04
Progress: 24 / 200 total iterations (12.250%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.894e+01  0.000e+00  y[m⁻³ kg] = 20.135[?]
3           4.142e+01  8.349e-02  y[m⁻³ kg] = radiation_mean[m⁻² W] * 0.1053[?]
4           4.018e+01  3.035e-02  y[m⁻³ kg] = sqrt(radiation_mean[m⁻² W]) * 1.4771[?...
                                              ]
5           3.950e+01  1.700e-02  y[m⁻³ kg] = (radiation_mean[m⁻² W] + 76.388[?]) * ...
                                              0.076376[?]
6           3.400e+01  1.499e-01  y[m⁻³ kg] = (sqrt(hotspots_sector_E_lag2) * 0.3454...
                                              8[?]) + 15.523[?]
7           3.392e+01  2.424e-03  y[m⁻³ kg] = sqrt((sqrt(hotspots_sector_E_lag2) - -...
       

┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


## Score every equation on every front: leave-one-year-out, constants refitted

Each numeric constant in the PySR expression becomes a free parameter, started from PySR's value. Evaluated on the same 2919 days
as Eq. 2 (rows with all of `temp_2m, no, pm25, w700, photochemical_potential, rh_2m` and `o3`).

In [4]:
FLOAT = re.compile(r'(?<![A-Za-z_0-9.])\d+\.\d+(?:[eE][+-]?\d+)?')
def make_model(eq, feats):
    vals = []
    def sub(m): vals.append(float(m.group(0))); return f"c{len(vals)-1}"
    s = FLOAT.sub(sub, eq)
    syms = {f: sp.Symbol(f) for f in feats}; cs = [sp.Symbol(f"c{i}") for i in range(len(vals))]
    expr = sp.sympify(s, locals={**syms, **{str(c): c for c in cs}, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    used = [f for f in feats if syms[f] in expr.free_symbols]
    keep = [i for i, c in enumerate(cs) if c in expr.free_symbols]
    fn = sp.lambdify([syms[f] for f in used] + [cs[i] for i in keep], expr, "numpy")
    p0 = np.array([vals[i] for i in keep])
    return (lambda Xt, *p: fn(*Xt, *p)), used, p0

O3_FEATS = ["no2", "no", "radiation_mean", "hotspots_sector_E_lag2", "rh_2m", "wind_speed_era5", "w700", "temp_2m", "pm25"]   # identical evaluation days for both pools
canon = df.dropna(subset=O3_FEATS + ['o3']).dropna(subset=features).reset_index(drop=True)
print("evaluation sample:", len(canon), "days (Eq. 2 uses 2919)")
yy = canon['o3'].to_numpy(float); years = canon['year'].to_numpy()

def score(eq):
    model, used, p0 = make_model(eq, features)
    if not used:                                   # constant equation (complexity 1): no variable, nothing to refit
        const = float(p0[0]) if len(p0) else float(eq)
        pred = np.full(len(yy), const)
        per = [r2_score(yy[years == yr], np.full((years == yr).sum(), np.mean(yy[years != yr]))) for yr in sorted(set(years))]
        oof = np.concatenate([np.full((years == yr).sum(), np.mean(yy[years != yr])) for yr in sorted(set(years))])
        yo = np.concatenate([yy[years == yr] for yr in sorted(set(years))])
        return dict(r2_insample=r2_score(yy, pred), r2_pooled=r2_score(yo, oof), r2_mean_years=float(np.mean(per)), vars="")
    Xt = tuple(canon[f].to_numpy(float) for f in used)
    with np.errstate(all='ignore'), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try: popt, _ = curve_fit(model, Xt, yy, p0=p0, maxfev=20000)
        except Exception: popt = p0
        pred = np.full(len(yy), np.nan)
        for yr in sorted(set(years)):
            te = years == yr
            try: pf, _ = curve_fit(model, tuple(a[~te] for a in Xt), yy[~te], p0=popt, maxfev=20000)
            except Exception: pf = popt
            pred[te] = model(tuple(a[te] for a in Xt), *pf)
        ok = np.all(np.isfinite(pred))
        ins = model(Xt, *popt)
        per = [r2_score(yy[years == yr], pred[years == yr]) for yr in sorted(set(years))] if ok else [np.nan]
        return dict(r2_insample=r2_score(yy, ins) if np.all(np.isfinite(ins)) else np.nan,
                    r2_pooled=r2_score(yy, pred) if ok else np.nan, r2_mean_years=float(np.mean(per)), vars=",".join(used))

rows = []
for seed, f in fronts.items():
    for _, r in f.iterrows():
        if r.complexity > MAX_COMPLEXITY_REPORTED: continue
        rows.append(dict(seed=seed, complexity=int(r.complexity), equation=r.equation, **score(r.equation)))
scored = pd.DataFrame(rows)
scored.to_csv(DATA_DIR / f"bogota_o3_pysr_{TAG}_scored.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 120)
print(scored[['seed','complexity','r2_pooled','r2_mean_years','r2_insample','vars']].round(3).to_string(index=False))

evaluation sample: 2917 days (Eq. 2 uses 2919)


 seed  complexity  r2_pooled  r2_mean_years  r2_insample                                                               vars
    0           1     -0.028         -0.149       -0.000                                                                   
    0           3      0.124          0.002        0.154                                                     radiation_mean
    0           4      0.151          0.042        0.179                                                     radiation_mean
    0           5      0.232          0.121        0.265                                             hotspots_sector_E_lag2
    0           6      0.274          0.162        0.307                                             hotspots_sector_E_lag2
    0           8      0.357          0.258        0.389                              radiation_mean,hotspots_sector_E_lag2
    0           9      0.369          0.269        0.401                              radiation_mean,hotspots_sector_E_lag2
    0   

## Reading the result
Compare, on the same days: free-form lead equation Eq. 2 (pooled 0.610, mean of years 0.543), the original constrained equation
without temperature (pooled 0.532, mean of years 0.448), and the best constrained equation here at the same complexity (16) and
overall. Look at whether `temp_2m` enters, with which partners, and whether `w700` is still needed.

In [5]:
summ = []
for seed in SEEDS:
    s = scored[scored.seed == seed]
    s16 = s[s.complexity <= 16].sort_values('r2_pooled', ascending=False).head(1)
    sb = s.sort_values('r2_pooled', ascending=False).head(1)
    for tag, d in (("best at complexity <=16", s16), ("best overall", sb)):
        if len(d): summ.append(dict(seed=seed, which=tag, complexity=int(d.complexity.iloc[0]), pooled=d.r2_pooled.iloc[0],
                                    mean_years=d.r2_mean_years.iloc[0], equation=d.equation.iloc[0]))
summary = pd.DataFrame(summ); summary.to_csv(DATA_DIR / f"bogota_o3_pysr_{TAG}_summary.csv", index=False)
print(summary.round(3).to_string(index=False))

 seed                   which  complexity  pooled  mean_years                                                                                                                                             equation
    0 best at complexity <=16          16   0.558       0.492                             sqrt(sqrt((0.06774135 - (sqrt(hotspots_sector_E_lag2) / (no * -4.301994))) * (no2 * (radiation_mean * radiation_mean))))
    0            best overall          18   0.589       0.523         sqrt(sqrt((no2 * (radiation_mean * radiation_mean)) * (0.07032524 - ((sqrt(hotspots_sector_E_lag2) / wind_speed_era5) / (no * -4.932103)))))
    1 best at complexity <=16          15   0.578       0.509                                    sqrt((hotspots_sector_E_lag2 / 3.8654706) - (-1.7302341 / (sqrt(no / (no2 / wind_speed_era5)) / radiation_mean)))
    1            best overall          21   0.584       0.520 sqrt(((hotspots_sector_E_lag2 / wind_speed_era5) / 3.8654706) - (-2.46608 / (sqrt(no / (no2 + 